# Chain Model Training with K-Fold - US Traffic Incident Analysis
ในขั้นตอนนี้เราจะเพิ่มการทำ **K-Fold Cross-Validation** เพื่อให้มั่นใจว่า Model ของเรามีความแม่นยำและเสถียร (Robust) ไม่ว่าจะสุ่มข้อมูลส่วนไหนมา Train ก็ตาม

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [2]:
df_train = pd.read_csv("../data/processed/03/train_features.csv")

# Bool columns from CSV stay as bool dtype — cast to int so XGBoost treats them as numeric
bool_cols = df_train.select_dtypes(include='bool').columns.tolist()
df_train[bool_cols] = df_train[bool_cols].astype(int)

print(f"Shape : {df_train.shape[0]:,} rows × {df_train.shape[1]} cols")
print(f"Bool→int : {bool_cols}")
df_train.head(3)

Shape : 5,469,092 rows × 61 cols
Bool→int : ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal']


,Severity,Start_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,...,Road_Complexity,Is_Intersection,Is_Controlled,State_Freq,City_Freq,is_shoulder,is_blocked,is_overturned,Rush_x_Severity,Night_x_Severity
0,3,2016-11-08 08:50:37,29.718655,-95.32105,0.00995,Accident on I-45 Northbound at I-45 Exits 43A ...,I-45 N,Houston,Harris,TX,...,1.098612,0.693147,0.693147,0.280078,0.022795,0.000000,0.000000,0.0,1.732051,0.0
1,2,2021-07-26 07:04:56,38.927550,-121.08009,0.00000,Right hand shoulder blocked due to accident on...,Taylor Ln,Auburn,Placer,CA,...,0.693147,0.000000,0.693147,0.473334,0.000813,0.693147,0.693147,0.0,1.414214,0.0
2,2,2018-10-24 08:23:38,34.776240,-86.67283,0.00000,Lane blocked due to accident on AL-255 Researc...,Highway 255,Huntsville,Madison,AL,...,0.000000,0.000000,0.000000,0.117871,0.001156,0.000000,0.693147,0.0,1.414214,0.0


## 2. Configuration & Features

In [3]:
TARGET_DIST = "Distance(mi)"
TARGET_DUR  = "Duration(min)"

# All text/categorical columns were encoded in feature engineering — nothing to one-hot
base_features = [c for c in df_train.columns if c not in [TARGET_DIST, TARGET_DUR]]
X_base        = df_train[base_features]

num_features = X_base.select_dtypes(include='number').columns.tolist()
cat_features = []   # already encoded in 03_feature_engineering

print(f"Base features : {len(base_features)}")
print(f"  Numerical   : {len(num_features)}  (cat=0, all encoded already)")
print(f"\n{num_features}")

Base features : 59
  Numerical   : 45  (cat=0, all encoded already)

['Severity', 'Start_Lat', 'Start_Lng', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Hour', 'Month', 'DayOfWeek', 'Is_Rush_Hour', 'Is_Weekend', 'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'Is_Night', 'Is_Bad_Weather', 'Low_Visibility_Flag', 'Freezing_Flag', 'Weather_Risk_Score', 'Road_Complexity', 'Is_Intersection', 'Is_Controlled', 'State_Freq', 'City_Freq', 'is_shoulder', 'is_blocked', 'is_overturned', 'Rush_x_Severity', 'Night_x_Severity']


## 3. K-Fold Cross-Validation Setup
เราจะใช้ 5-Fold Cross-Validation เพื่อทดสอบประสิทธิภาพ

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def run_kfold_cv(model, X, y, name):
    print(f"Running 5-Fold CV for {name}...")
    # เพื่อความรวดเร็วในการรัน K-Fold กับข้อมูล 5 ล้านแถว 
    # จะขอสุ่มมาใช้เพียง 10% (5 แสนแถว) สำหรับการทำ CV
    sample_idx = np.random.choice(len(X), int(len(X)*0.1), replace=False)
    X_sample = X.iloc[sample_idx]
    y_sample = y.iloc[sample_idx]
    
    scores = cross_val_score(model, X_sample, y_sample, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    print(f"{name} CV MAE: {mae_scores.mean():.4f} (+/- {mae_scores.std():.4f})")
    return mae_scores

NameError: name 'KFold' is not defined

## 4. Model 1 (Distance) with CV

In [ ]:
y_distance = df_train[TARGET_DIST]

# Split — save indices so Model 2 uses the exact same rows
X_train_d, X_val_d, y_train_d, y_val_d = train_test_split(
    X_base, y_distance, test_size=0.2, random_state=42
)
TRAIN_IDX = X_train_d.index
VAL_IDX   = X_val_d.index

# Preprocessor: impute missing + standardise
preprocessor_dist = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), num_features)
], remainder='drop')

preprocessor_dist.fit(X_train_d)
X_tr_d = preprocessor_dist.transform(X_train_d)
X_vl_d_proc = preprocessor_dist.transform(X_val_d)

# XGBoost with early stopping — stops when val MAE doesn't improve for 20 rounds
xgb_dist = XGBRegressor(
    n_estimators        = 500,
    learning_rate       = 0.05,
    max_depth           = 7,
    min_child_weight    = 5,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    reg_alpha           = 0.1,
    reg_lambda          = 1.0,
    random_state        = 42,
    tree_method         = 'hist',
    early_stopping_rounds = 20,
    eval_metric         = 'mae',
    n_jobs              = -1,
)

print("Training Model 1 (Distance)...")
xgb_dist.fit(
    X_tr_d, y_train_d,
    eval_set=[(X_vl_d_proc, y_val_d)],
    verbose=50
)

# Wrap back into a Pipeline for consistent .predict(X_raw) interface
model_dist = Pipeline([('preprocessor', preprocessor_dist), ('regressor', xgb_dist)])

pred_d = xgb_dist.predict(X_vl_d_proc)
print(f"\n=== Model 1 — Distance Predictor ===")
print(f"  MAE       : {mean_absolute_error(y_val_d, pred_d):.4f}")
print(f"  RMSE      : {np.sqrt(mean_squared_error(y_val_d, pred_d)):.4f}")
print(f"  R²        : {r2_score(y_val_d, pred_d):.4f}")
print(f"  Best iter : {xgb_dist.best_iteration}")

## 5. Model 2 (Duration) with CV

In [ ]:
y_duration = df_train[TARGET_DUR]

# Model 2 needs Distance(mi) as an extra feature
# Training  → ground-truth distance (teacher forcing)
# Validation → two versions: oracle (GT) vs chain (Model 1 prediction)
num_features_dur = num_features + [TARGET_DIST]

X_with_dist_gt = X_base.copy()
X_with_dist_gt[TARGET_DIST] = df_train[TARGET_DIST]

X_train_t    = X_with_dist_gt.loc[TRAIN_IDX]
y_train_t    = y_duration.loc[TRAIN_IDX]
y_val_t      = y_duration.loc[VAL_IDX]

# Oracle val: uses actual Distance (upper bound performance)
X_val_oracle = X_with_dist_gt.loc[VAL_IDX].copy()

# Chain val: inject Model 1's predicted Distance (realistic inference)
X_val_chain               = X_val_d.copy()
X_val_chain[TARGET_DIST]  = pred_d        # Model 1 output

# Preprocessor
preprocessor_dur = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), num_features_dur)
], remainder='drop')

preprocessor_dur.fit(X_train_t)
X_tr_t       = preprocessor_dur.transform(X_train_t)
X_vl_oracle  = preprocessor_dur.transform(X_val_oracle)
X_vl_chain   = preprocessor_dur.transform(X_val_chain)

xgb_dur = XGBRegressor(
    n_estimators        = 500,
    learning_rate       = 0.05,
    max_depth           = 7,
    min_child_weight    = 5,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    reg_alpha           = 0.1,
    reg_lambda          = 1.0,
    random_state        = 42,
    tree_method         = 'hist',
    early_stopping_rounds = 20,
    eval_metric         = 'mae',
    n_jobs              = -1,
)

print("Training Model 2 (Duration)...")
xgb_dur.fit(
    X_tr_t, y_train_t,
    eval_set=[(X_vl_oracle, y_val_t)],   # early stop on oracle
    verbose=50
)

model_dur = Pipeline([('preprocessor', preprocessor_dur), ('regressor', xgb_dur)])

pred_dur_oracle = xgb_dur.predict(X_vl_oracle)
pred_dur_chain  = xgb_dur.predict(X_vl_chain)

print(f"\n=== Model 2 — Duration Predictor ===")
print("  Oracle (GT Distance — upper bound):")
print(f"    MAE  : {mean_absolute_error(y_val_t, pred_dur_oracle):.4f}")
print(f"    RMSE : {np.sqrt(mean_squared_error(y_val_t, pred_dur_oracle)):.4f}")
print(f"    R²   : {r2_score(y_val_t, pred_dur_oracle):.4f}")
print("  Chain (Predicted Distance — realistic):")
print(f"    MAE  : {mean_absolute_error(y_val_t, pred_dur_chain):.4f}")
print(f"    RMSE : {np.sqrt(mean_squared_error(y_val_t, pred_dur_chain)):.4f}")
print(f"    R²   : {r2_score(y_val_t, pred_dur_chain):.4f}")
print(f"  Best iter : {xgb_dur.best_iteration}")

In [ ]:
rng   = np.random.default_rng(42)
idx_s = rng.choice(len(y_val_d), min(5000, len(y_val_d)), replace=False)

def scatter_eval(ax, y_true, y_pred, title, color):
    ax.scatter(y_true.iloc[idx_s], y_pred[idx_s], alpha=0.25, s=4, color=color)
    lo = min(float(y_true.min()), float(y_pred.min()))
    hi = max(float(y_true.max()), float(y_pred.max()))
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    ax.set_title(f'{title}\nMAE={mae:.4f}   R²={r2:.3f}', fontsize=11)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
scatter_eval(axes[0], y_val_d, pred_d,            'Model 1: Distance(mi)',           '#1565C0')
scatter_eval(axes[1], y_val_t, pred_dur_oracle,   'Model 2: Duration — Oracle Dist', '#2E7D32')
scatter_eval(axes[2], y_val_t, pred_dur_chain,    'Model 2: Duration — Chain Dist',  '#E65100')
plt.suptitle('Chain Model — Predicted vs Actual', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for i, (ax, label, xgb_model, feat_list) in enumerate([
    (axes[0], 'Model 1 — Distance',  xgb_dist, num_features),
    (axes[1], 'Model 2 — Duration',  xgb_dur,  num_features_dur),
]):
    importance = pd.Series(xgb_model.feature_importances_, index=feat_list)
    top20 = importance.sort_values(ascending=False).head(20)
    ax.barh(top20.index[::-1], top20.values[::-1], color='steelblue', edgecolor='white')
    ax.set_title(f'{label}\nTop 20 Feature Importance (gain)', fontsize=12)
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance — Chain Model', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Save Models

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(model_dist, "../models/chain_dist_model.pkl")
joblib.dump(model_dur,  "../models/chain_dur_model.pkl")

# Save feature metadata for inference
metadata = {
    "base_features"    : base_features,
    "num_features"     : num_features,
    "num_features_dur" : num_features_dur,
    "target_dist"      : TARGET_DIST,
    "target_dur"       : TARGET_DUR,
}
joblib.dump(metadata, "../models/chain_metadata.pkl")

print("Saved:")
print(f"  models/chain_dist_model.pkl   ({len(num_features)} features)")
print(f"  models/chain_dur_model.pkl    ({len(num_features_dur)} features)")
print(f"  models/chain_metadata.pkl")

# Final summary table
rows = [
    ["Distance", "—",      f"{mean_absolute_error(y_val_d, pred_d):.4f}",
                           f"{np.sqrt(mean_squared_error(y_val_d, pred_d)):.4f}",
                           f"{r2_score(y_val_d, pred_d):.4f}"],
    ["Duration", "Oracle", f"{mean_absolute_error(y_val_t, pred_dur_oracle):.4f}",
                           f"{np.sqrt(mean_squared_error(y_val_t, pred_dur_oracle)):.4f}",
                           f"{r2_score(y_val_t, pred_dur_oracle):.4f}"],
    ["Duration", "Chain",  f"{mean_absolute_error(y_val_t, pred_dur_chain):.4f}",
                           f"{np.sqrt(mean_squared_error(y_val_t, pred_dur_chain)):.4f}",
                           f"{r2_score(y_val_t, pred_dur_chain):.4f}"],
]
print("\n=== Final Chain Model Summary ===")
print(pd.DataFrame(rows, columns=["Target","Mode","MAE","RMSE","R²"]).to_string(index=False))